# 03: 0D Windkessel Model Formulation & Sensitivity Analysis (Step 4)
**Project:** DigiTwin - AI-Powered Personalized Digital Twin for Cardiovascular Treatment Simulation
**Academic Context:** V-Semester B.E. CSE (AI&ML), SIT Tumakuru
**Disclaimer:** Academic research prototype only. NOT a clinical tool; do not use for actual patient medication.

---
## 1. Mathematical Formulation & State Variables
The lumped 0D arterial hemodynamics are modeled using a 3-element Windkessel ($R-C-R_c$) formulation:

$$\frac{dP_c(t)}{dt} = \frac{Q(t) - P_c(t)/R}{C}$$
$$P_a(t) = P_c(t) + R_c \cdot Q(t)$$

* **$P_c(t)$ (State Variable):** Blood pressure across the total arterial compliance reservoir ($C$) (mmHg).
* **$P_a(t)$:** Aortic root pressure (mmHg).
* **$R$:** Total systemic peripheral vascular resistance (mmHg*s/ml).
* **$C$:** Total systemic arterial compliance (ml/mmHg).
* **$R_c$:** Characteristic aortic input impedance (mmHg*s/ml), default 0.05.
* **$Q(t)$:** Inflow rate from left ventricular ejection (ml/s):
  $$Q(t) = \begin{cases} \frac{\pi \cdot SV}{2 t_s} \sin\left(\frac{\pi t}{t_s}\right) & 0 \le t \le t_s \\ 0 & t_s < t < T \end{cases}$$
* **Cardiac timing:** Period $T = 60 / HR$ (s), systolic ejection duration $t_s = 0.35 \sqrt{60/HR}$ (s).
* **Periodic steady state ($x_0$):** Solved in closed form via linear superposition over systole combined with exponential diastolic decay ($e^{-t / (RC)}$).

## 2. Parameter Classifications and Literature Citations

| Parameter | Units | Value / Range | Label | Literature Source |
|---|---|---|---|---|
| $R$ | mmHg*s/ml | Calibrated per patient ($0.6 - 1.8$) | **Estimated** | Least-squares fitting on measured SBP/DBP |
| $C$ | ml/mmHg | Calibrated per patient ($0.8 - 2.2$) | **Estimated** | Simultaneous calibration with $R$ |
| $R_c$ | mmHg*s/ml | 0.05 | **Literature** | Westerhof N, et al. *An artificial arterial system for pumping hearts.* J Appl Physiol. 1971;31(5):776-81. |
| Waveform $Q(t)$ | ml/s | Half-sine, $\int Q dt = SV$ | **Literature** | Stergiopulos N, et al. *Evaluation of methods for estimation of total arterial compliance.* Am J Physiol. 1995;268(4):H1540-8. |
| Ejection $t_s$ | s | $0.35 \sqrt{60/HR}$ | **Literature** | Weissler AM, et al. *Systolic time intervals in heart failure in man.* Circulation. 1968;37(2):149-59. |


In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd() / 'backend')) if (Path.cwd() / 'backend').exists() else sys.path.insert(0, str(Path('..') / 'backend'))
from app.physics.windkessel import simulate_beat, calibrate, ejection_time

print("Running solver validation and physical consistency tests across 100 parameter points...")
passed = 0
for R in np.linspace(0.4, 2.5, 5):
    for C in np.linspace(0.4, 2.5, 5):
        for hr in (50, 75, 100, 130):
            sv = 65.0
            beat = simulate_beat(R, C, hr, sv)
            # Physical sanity checks:
            assert np.isfinite(beat.sbp) and np.isfinite(beat.dbp) and np.isfinite(beat.map)
            assert beat.sbp > beat.map > beat.dbp > 0
            # Flow conservation: MAP == (CO)*(R + Rc)
            expected_map = (hr * sv / 60.0) * (R + 0.05)
            assert abs(beat.map - expected_map) < 1e-4
            passed += 1
print(f"All {passed} solver sanity checks passed: outputs are finite, positive, and satisfy flow conservation.")


Running solver validation and physical consistency tests across 100 parameter points...
All 100 solver sanity checks passed: outputs are finite, positive, and satisfy flow conservation.


In [2]:
# Sensitivity Plots: SBP, DBP, MAP, PP against R and C
R_curves = [0.8, 1.1, 1.4, 1.8]
C_vals = np.linspace(0.5, 2.5, 25)
hr_ref, sv_ref = 72.0, 65.0

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
colors = ["#2563eb", "#059669", "#d97706", "#dc2626"]

for idx, R in enumerate(R_curves):
    sbp_list = [simulate_beat(R, C, hr_ref, sv_ref).sbp for C in C_vals]
    dbp_list = [simulate_beat(R, C, hr_ref, sv_ref).dbp for C in C_vals]
    pp_list = [s - d for s, d in zip(sbp_list, dbp_list)]
    axes[0, 0].plot(C_vals, sbp_list, color=colors[idx], lw=2, label=f"R = {R} mmHg*s/ml")
    axes[0, 1].plot(C_vals, dbp_list, color=colors[idx], lw=2, label=f"R = {R} mmHg*s/ml")
    axes[1, 0].plot(C_vals, pp_list, color=colors[idx], lw=2, label=f"R = {R} mmHg*s/ml")

axes[0, 0].set_title("Systolic Blood Pressure (SBP) vs Compliance C")
axes[0, 0].set_xlabel("Arterial Compliance C (ml/mmHg)")
axes[0, 0].set_ylabel("SBP (mmHg)")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].set_title("Diastolic Blood Pressure (DBP) vs Compliance C")
axes[0, 1].set_xlabel("Arterial Compliance C (ml/mmHg)")
axes[0, 1].set_ylabel("DBP (mmHg)")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].set_title("Pulse Pressure (PP = SBP - DBP) vs Compliance C")
axes[1, 0].set_xlabel("Arterial Compliance C (ml/mmHg)")
axes[1, 0].set_ylabel("Pulse Pressure (mmHg)")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Subplot 4: MAP vs R for different C
R_range = np.linspace(0.6, 2.2, 25)
for idx, C in enumerate([0.8, 1.2, 1.6, 2.0]):
    map_list = [simulate_beat(R, C, hr_ref, sv_ref).map for R in R_range]
    axes[1, 1].plot(R_range, map_list, color=colors[idx], lw=2, ls="--", label=f"C = {C} ml/mmHg")
axes[1, 1].set_title("Mean Arterial Pressure (MAP) vs Resistance R")
axes[1, 1].set_xlabel("Peripheral Resistance R (mmHg*s/ml)")
axes[1, 1].set_ylabel("MAP (mmHg)")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
out_img = Path('docs/windkessel_sensitivity.png') if Path('docs').exists() else Path('../docs/windkessel_sensitivity.png')
plt.savefig(out_img, dpi=150)
plt.close()
print(f'Sensitivity plots successfully saved to {out_img}')


Sensitivity plots successfully saved to docs\windkessel_sensitivity.png


In [3]:
# Normalized Sensitivity Coefficients (Elasticities) around baseline (R=1.0, C=1.5)
R0, C0 = 1.0, 1.5
base = simulate_beat(R0, C0, hr_ref, sv_ref)
eps = 1e-4

# Partial w.r.t R
bR_hi = simulate_beat(R0 * (1 + eps), C0, hr_ref, sv_ref)
bR_lo = simulate_beat(R0 * (1 - eps), C0, hr_ref, sv_ref)
d_sbp_dR = (bR_hi.sbp - bR_lo.sbp) / (2 * eps * R0)
d_dbp_dR = (bR_hi.dbp - bR_lo.dbp) / (2 * eps * R0)
d_map_dR = (bR_hi.map - bR_lo.map) / (2 * eps * R0)
d_pp_dR  = ((bR_hi.sbp - bR_hi.dbp) - (bR_lo.sbp - bR_lo.dbp)) / (2 * eps * R0)

# Partial w.r.t C
bC_hi = simulate_beat(R0, C0 * (1 + eps), hr_ref, sv_ref)
bC_lo = simulate_beat(R0, C0 * (1 - eps), hr_ref, sv_ref)
d_sbp_dC = (bC_hi.sbp - bC_lo.sbp) / (2 * eps * C0)
d_dbp_dC = (bC_hi.dbp - bC_lo.dbp) / (2 * eps * C0)
d_map_dC = (bC_hi.map - bC_lo.map) / (2 * eps * C0)
d_pp_dC  = ((bC_hi.sbp - bC_hi.dbp) - (bC_lo.sbp - bC_lo.dbp)) / (2 * eps * C0)

df_sens = pd.DataFrame({
    "Variable": ["SBP", "DBP", "MAP", "PP"],
    "Baseline (mmHg)": [round(base.sbp, 1), round(base.dbp, 1), round(base.map, 1), round(base.sbp - base.dbp, 1)],
    "d/dR (mmHg / [mmHg*s/ml])": [round(d_sbp_dR, 2), round(d_dbp_dR, 2), round(d_map_dR, 2), round(d_pp_dR, 2)],
    "Elasticity w.r.t R": [round((R0 / base.sbp) * d_sbp_dR, 3), round((R0 / base.dbp) * d_dbp_dR, 3), round((R0 / base.map) * d_map_dR, 3), round((R0 / (base.sbp - base.dbp)) * d_pp_dR, 3)],
    "d/dC (mmHg / [ml/mmHg])": [round(d_sbp_dC, 2), round(d_dbp_dC, 2), round(d_map_dC, 2), round(d_pp_dC, 2)],
    "Elasticity w.r.t C": [round((C0 / base.sbp) * d_sbp_dC, 3), round((C0 / base.dbp) * d_dbp_dC, 3), round((C0 / base.map) * d_map_dC, 3), round((C0 / (base.sbp - base.dbp)) * d_pp_dC, 3)]
})
print("\n--- Local Sensitivity Table (R=1.0 mmHg*s/ml, C=1.5 ml/mmHg, HR=72, SV=65) ---")
print(df_sens.to_string(index=False))



--- Local Sensitivity Table (R=1.0 mmHg*s/ml, C=1.5 ml/mmHg, HR=72, SV=65) ---
Variable  Baseline (mmHg)  d/dR (mmHg / [mmHg*s/ml])  Elasticity w.r.t R  d/dC (mmHg / [ml/mmHg])  Elasticity w.r.t C
     SBP            101.4                      77.15               0.761                    -8.16              -0.121
     DBP             65.0                      77.60               1.195                     8.43               0.195
     MAP             81.9                      78.00               0.952                     0.00               0.000
      PP             36.5                      -0.45              -0.012                   -16.59              -0.683


In [4]:
# Calibration repeatability verification
print("\n--- Calibration Repeatability on Clinical Targets ---")
targets = [(120, 80, 72, 65), (150, 95, 80, 60), (95, 60, 65, 70), (170, 75, 68, 65)]
for target_sbp, target_dbp, test_hr, test_sv in targets:
    R_fit, C_fit = calibrate(target_sbp, target_dbp, test_hr, test_sv)
    b_recov = simulate_beat(R_fit, C_fit, test_hr, test_sv)
    err_sbp = abs(b_recov.sbp - target_sbp)
    err_dbp = abs(b_recov.dbp - target_dbp)
    print(f"Target ({target_sbp}/{target_dbp}) -> Calibrated R={R_fit:.3f}, C={C_fit:.3f} -> Sim ({b_recov.sbp:.2f}/{b_recov.dbp:.2f}) | Error: SBP {err_sbp:.4f} mmHg, DBP {err_dbp:.4f} mmHg")
    assert err_sbp < 0.1 and err_dbp < 0.1, "Calibration error exceeded!"
print("All calibration cases recovered within 0.1 mmHg tolerance.")



--- Calibration Repeatability on Clinical Targets ---
Target (120/80) -> Calibrated R=1.218, C=1.310 -> Sim (120.00/80.00) | Error: SBP 0.0000 mmHg, DBP 0.0000 mmHg
Target (150/95) -> Calibrated R=1.469, C=0.775 -> Sim (150.00/95.00) | Error: SBP 0.0000 mmHg, DBP 0.0000 mmHg
Target (95/60) -> Calibrated R=0.951, C=1.782 -> Sim (95.00/60.00) | Error: SBP 0.0000 mmHg, DBP 0.0000 mmHg
Target (170/75) -> Calibrated R=1.568, C=0.468 -> Sim (170.00/75.00) | Error: SBP 0.0000 mmHg, DBP 0.0000 mmHg
All calibration cases recovered within 0.1 mmHg tolerance.
